<a href="https://colab.research.google.com/github/nikimajithiya83/AI_lab_experiments/blob/main/Collaborative_based_recommendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("luisreimberg/ratingscsv")

print("Path to dataset files:", path)

# Load the ratings data from the downloaded path
ratings = pd.read_csv(os.path.join(path, 'ratings.csv'))

# ================= USER-ITEM MATRIX =================
user_item_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
).fillna(0)
# ================= USER SIMILARITY =================
user_similarity = cosine_similarity(user_item_matrix)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)
# ================= RECOMMEND FUNCTION =================
def recommend_movies(user_id, top_n=5):
    if user_id not in user_item_matrix.index:
        return "User not found!"

    # Similar users
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)[1:6]

    # Weighted ratings
    weighted_ratings = np.zeros(user_item_matrix.shape[1])

    for sim_user, score in similar_users.items():
        weighted_ratings += score * user_item_matrix.loc[sim_user].values

    # Normalize
    weighted_ratings /= similar_users.sum()

    # Get unseen movies
    user_rated = user_item_matrix.loc[user_id]
    unseen_movies = user_rated[user_rated == 0].index

    scores = list(zip(unseen_movies, weighted_ratings[user_item_matrix.columns.get_indexer(unseen_movies)]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    recommended_ids = [movie for movie, _ in scores[:top_n]]

    return recommended_ids
print(recommend_movies(user_id=1))

Using Colab cache for faster access to the 'ratingscsv' dataset.
Path to dataset files: /kaggle/input/ratingscsv
[1200, 1610, 541, 589, 1036]


Postlab

In [6]:
# ================= INSTALL & DOWNLOAD DATA =================
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Download dataset (works in Colab)
import os

if not os.path.exists("ml-100k"):
    !wget https://files.grouplens.org/datasets/movielens/ml-100k.zip
    !unzip ml-100k.zip

# ================= LOAD DATA =================
ratings = pd.read_csv("ml-100k/u.data",
                      sep='\t',
                      names=['userId','movieId','rating','timestamp'])

movies = pd.read_csv("ml-100k/u.item",
                     sep='|',
                     encoding='latin-1',
                     header=None)

movies = movies[[0,1]]
movies.columns = ['movieId','title']

# ================= USER-ITEM MATRIX =================
user_item = ratings.pivot_table(index='userId',
                               columns='movieId',
                               values='rating').fillna(0)

# ================= USER SIMILARITY =================
similarity = cosine_similarity(user_item)

similarity_df = pd.DataFrame(similarity,
                            index=user_item.index,
                            columns=user_item.index)

# ================= RECOMMEND FUNCTION =================
def recommend_movies(user_id, top_n=10):

    if user_id not in user_item.index:
        return "User not found!"

    # Find similar users
    similar_users = similarity_df[user_id].sort_values(ascending=False)[1:6]

    # Compute weighted ratings
    weighted_ratings = np.zeros(user_item.shape[1])

    for sim_user, score in similar_users.items():
        weighted_ratings += score * user_item.loc[sim_user].values

    # Normalize
    weighted_ratings /= similar_users.sum()

    # Movies not watched by user
    user_rated = user_item.loc[user_id]
    unseen_movies = user_rated[user_rated == 0].index

    # Score unseen movies
    scores = list(zip(unseen_movies,
                      weighted_ratings[user_item.columns.get_indexer(unseen_movies)]))

    # Sort movies by score
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    # Top N movie IDs
    recommended_ids = [movie for movie, _ in scores[:top_n]]

    # Get movie names
    recommended_movies = movies[movies['movieId'].isin(recommended_ids)]

    return recommended_movies

# ================= OUTPUT =================
user_id = 1
print("Top 10 Recommended Movies:\n")
print(recommend_movies(user_id, top_n=10))

--2026-04-14 13:45:57--  https://files.grouplens.org/datasets/movielens/ml-100k.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4924029 (4.7M) [application/zip]
Saving to: ‘ml-100k.zip’

ml-100k.zip         100%[===================>]   4.70M  21.9MB/s    in 0.2s    

2026-04-14 13:45:58 (21.9 MB/s) - ‘ml-100k.zip’ saved [4924029/4924029]

Archive:  ml-100k.zip
   creating: ml-100k/
  inflating: ml-100k/allbut.pl       
  inflating: ml-100k/mku.sh          
  inflating: ml-100k/README          
  inflating: ml-100k/u.data          
  inflating: ml-100k/u.genre         
  inflating: ml-100k/u.info          
  inflating: ml-100k/u.item          
  inflating: ml-100k/u.occupation    
  inflating: ml-100k/u.user          
  inflating: ml-100k/u1.base         
  inflating: ml-100k/u1.test         
  inflating: ml-100k/u2.ba